**<h1 align="center">ESCO Feature Construction</h1>**

This notebook constructs ontology-based feature artifacts from the ESCO v1.1 dataset to enable skill-aware job representation. It focuses on essential skill relations to build structured mappings between occupations and skills and to compute corpus-level statistics.

The notebook derives key artifacts, including occupation–skill mappings, skill document frequency, inverse document frequency (IDF) weights, and occupation-level skill specificity scores. All outputs are saved as reusable JSON files for downstream experiments.

# Table of Contents
* [1. Imports & Setup](#chapter1)
* [2. Load Data](#chapter2)
* [3. Build Lookup Tables](#chapter3)
* [4. Build Skill Hierarchy Lookup (skill → parents)](#chapter4)
* [5. Build Occupation → Essential Skills Map](#chapter5)
* [6. Compute IDF for Skills (global specificity) & Occupation-Level Specificity (avg skill IDF)](#chapter6)
* [7. Coverage + Diagnostics](#chapter7)
* [8. Save Artifacts](#chapter8)


<a class="anchor" id="chapter1"></a>

# 1. Imports & Setup

</a>

In [2]:
import os
import json
import re
import sys
from pathlib import Path
import math
import pandas as pd
from collections import defaultdict, Counter

In [3]:
# Project root (assumes notebook is in /Notebook)
PROJECT_ROOT = Path("..").resolve()

# Allow imports from project root
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [4]:
from thesis_utility import save_json

/opt/anaconda3/envs/ResumeMatcherThesis/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<a class="anchor" id="chapter2"></a>

# 2. Load Data

</a>

In [7]:
DATA_DIR = PROJECT_ROOT / "Data" / "ESCO Files"

In [8]:
dictionary = pd.read_csv(DATA_DIR / "dictionary_en.csv")
skills = pd.read_csv(DATA_DIR / "skills_en.csv")
occ_skill = pd.read_csv(DATA_DIR / "occupationSkillRelations_en.csv")
occ = pd.read_csv(DATA_DIR / "occupations_en.csv")
hierarchy = pd.read_csv(DATA_DIR / "skillsHierarchy_en.csv")

<a class="anchor" id="chapter3"></a>

# 3. Build Lookup Tables

</a>

In [38]:
print("skills_en.csv:", skills.columns)
print("occupationSkillRelations_en.csv:", occ_skill.columns)
print("occupations_en.csv:", occ.columns)

skills_en.csv: Index(['conceptType', 'conceptUri', 'skillType', 'reuseLevel',
       'preferredLabel', 'altLabels', 'hiddenLabels', 'status', 'modifiedDate',
       'scopeNote', 'definition', 'inScheme', 'description'],
      dtype='object')
occupationSkillRelations_en.csv: Index(['occupationUri', 'occupationLabel', 'relationType', 'skillType',
       'skillUri', 'skillLabel'],
      dtype='object')
occupations_en.csv: Index(['conceptType', 'conceptUri', 'iscoGroup', 'preferredLabel', 'altLabels',
       'hiddenLabels', 'status', 'modifiedDate', 'regulatedProfessionNote',
       'scopeNote', 'definition', 'inScheme', 'description', 'code',
       'naceCode'],
      dtype='object')


In [39]:
# Filter occupation–skill relations to essential skills only
essential_skills = occ_skill[occ_skill['relationType'] == 'essential']

In [40]:
# Build lookup dictionaries for occupations (URI → label, ISCO group, description)
occ_to_label = occ.set_index('conceptUri')['preferredLabel'].to_dict()
occ_to_isco = occ.set_index('conceptUri')['iscoGroup'].to_dict()
occ_to_desc = occ.set_index('conceptUri')['description'].to_dict()

In [41]:
# Build lookup dictionaries for skills (URI → label, skill type)
skill_to_label = skills.set_index('conceptUri')['preferredLabel'].to_dict()
skill_to_type = skills.set_index('conceptUri')['skillType'].to_dict()

<a class="anchor" id="chapter4"></a>

# 4. Build Skill Hierarchy Lookup (skill → parents)

</a>

In [42]:
dictionary[dictionary["filename"] == "skillsHierarchy"]

,filename,data header,property,description
146,skillsHierarchy,Level 0 URI,NaN,No direct related property exists
147,skillsHierarchy,Level 0 preferred term,http://www.w3.org/2004/02/skos/core#prefLabel,"The preferred lexical label for a resource, in..."
148,skillsHierarchy,Level 1 URI,NaN,No direct related property exists
149,skillsHierarchy,Level 1 preferred term,http://www.w3.org/2004/02/skos/core#prefLabel,"The preferred lexical label for a resource, in..."
150,skillsHierarchy,Level 2 URI,NaN,No direct related property exists
151,skillsHierarchy,Level 2 preferred term,http://www.w3.org/2004/02/skos/core#prefLabel,"The preferred lexical label for a resource, in..."
152,skillsHierarchy,Level 3 URI,NaN,No direct related property exists
153,skillsHierarchy,Level 3 preferred term,http://www.w3.org/2004/02/skos/core#prefLabel,"The preferred lexical label for a resource, in..."
154,skillsHierarchy,Description,http://purl.org/dc/terms/description,An account of the resource.
155,skillsHierarchy,Scope note,http://www.w3.org/2004/02/skos/core#scopeNote,A note that helps to clarify the meaning and/o...


The ESCO skill hierarchy provides paths from Level 0 to Level 3. These paths are converted into direct parent–child relationships, where each level is linked to its immediate parent (e.g., Level 2 → Level 1). The resulting mapping defines, for each skill, its direct parent skills in the hierarchy.

In [43]:
# Build mapping from each skill to its direct parent(s) based on hierarchy levels
def build_skill_to_parents_from_levels(hierarchy_df: pd.DataFrame) -> dict:
    parent_map = defaultdict(set)

    # Ensure required hierarchy columns are present
    required = ["Level 0 URI", "Level 1 URI", "Level 2 URI", "Level 3 URI"]
    missing = [c for c in required if c not in hierarchy_df.columns]
    if missing:
        raise ValueError(f"skillsHierarchy_en.csv is missing columns: {missing}")

    # Iterate over hierarchy paths and create parent-child links
    for _, row in hierarchy_df.iterrows():
        l0 = row["Level 0 URI"]
        l1 = row["Level 1 URI"]
        l2 = row["Level 2 URI"]
        l3 = row["Level 3 URI"]

        # Add direct parent relationships between consecutive levels
        if isinstance(l0, str) and isinstance(l1, str) and l0.strip() and l1.strip():
            parent_map[l1.strip()].add(l0.strip())

        if isinstance(l1, str) and isinstance(l2, str) and l1.strip() and l2.strip():
            parent_map[l2.strip()].add(l1.strip())

        if isinstance(l2, str) and isinstance(l3, str) and l2.strip() and l3.strip():
            parent_map[l3.strip()].add(l2.strip())

    # Convert to dictionary format (child → list of parents)
    skill_to_parents = {child: sorted(list(parents)) for child, parents in parent_map.items()}
    return skill_to_parents

In [44]:
skill_to_parents = build_skill_to_parents_from_levels(hierarchy)

In [45]:
print("Skills with >=1 parent:", len(skill_to_parents))

Skills with >=1 parent: 636


<a class="anchor" id="chapter5"></a>

# 5. Build Occupation → Essential Skills Map

</a>

In [46]:
# Build mapping from occupation URI to set of essential skill URIs
occ_to_skills = defaultdict(set)
for _, row in essential_skills.iterrows():
    occ_to_skills[row['occupationUri']].add(row['skillUri'])

In [47]:
# Compute number of essential skills per occupation
occ_skill_count = {occ: len(skills) for occ, skills in occ_to_skills.items()}

<a class="anchor" id="chapter6"></a>

# 6. Compute IDF for Skills (global specificity) & Occupation-Level Specificity (avg skill IDF)

</a>

In [48]:
# occ_to_skills: dict[occ_uri] -> set[skill_uri]
N_occ = len(occ_to_skills)

# Compute document frequency: number of occupations each skill appears in
df_counter = Counter()
for skills_set in occ_to_skills.values():
    df_counter.update(skills_set)

skill_df = dict(df_counter)

# Compute inverse document frequency (IDF) for each skill (higher IDF = more specific / less frequent skill)
skill_idf = {s: math.log((N_occ + 1) / (df + 1)) + 1.0 for s, df in skill_df.items()}

# Compute occupation-level skill specificity (average IDF of its skills)
occ_avg_skill_idf = {
    occ: (sum(skill_idf[s] for s in skills_set) / max(1, len(skills_set)))
    for occ, skills_set in occ_to_skills.items()
}

<a class="anchor" id="chapter7"></a>

# 7. Coverage + Diagnostics

</a>

In [49]:
n_occ_total = occ['conceptUri'].nunique()
n_occ_with_skills = len(occ_to_skills)

n_skills_total = skills['conceptUri'].nunique()
n_skills_in_mapping = len(skill_df)

print("Occupations total:", n_occ_total)
print("Occupations with >=1 essential skill:", n_occ_with_skills,
      f"({n_occ_with_skills/n_occ_total:.2%})")

print("Skills total:", n_skills_total)
print("Skills appearing as essential at least once:", n_skills_in_mapping,
      f"({n_skills_in_mapping/n_skills_total:.2%})")

Occupations total: 3039
Occupations with >=1 essential skill: 3039 (100.00%)
Skills total: 13939
Skills appearing as essential at least once: 11378 (81.63%)


In [50]:
occ_skill_counts = [len(s) for s in occ_to_skills.values()]
print("Essential skills per occupation:")
print("min / median / mean / max:",
      min(occ_skill_counts),
      sorted(occ_skill_counts)[len(occ_skill_counts)//2],
      sum(occ_skill_counts)/len(occ_skill_counts),
      max(occ_skill_counts))

Essential skills per occupation:
min / median / mean / max: 3 19 22.24415926291543 99


In [51]:
# occupations referenced in relations but missing from occupations file
occ_set = set(occ['conceptUri'])
mapped_occ_set = set(occ_to_skills.keys())
missing_occ = mapped_occ_set - occ_set
print("Occupations in relations but not in occupations_en:", len(missing_occ))

# skills referenced in relations but missing from skills file
skill_set = set(skills['conceptUri'])
mapped_skill_set = set(skill_df.keys())
missing_skill = mapped_skill_set - skill_set
print("Skills in relations but not in skills_en:", len(missing_skill))

Occupations in relations but not in occupations_en: 0
Skills in relations but not in skills_en: 0


In [52]:
n_skill_nodes_in_hierarchy = len(set(hierarchy["Level 0 URI"].dropna().astype(str))
                                 | set(hierarchy["Level 1 URI"].dropna().astype(str))
                                 | set(hierarchy["Level 2 URI"].dropna().astype(str))
                                 | set(hierarchy["Level 3 URI"].dropna().astype(str)))

print("Unique skill URIs appearing anywhere in hierarchy:", n_skill_nodes_in_hierarchy)
print("Skills that have at least one parent (child->parent edges):", len(skill_to_parents))

Unique skill URIs appearing anywhere in hierarchy: 640
Skills that have at least one parent (child->parent edges): 636


<a class="anchor" id="chapter8"></a>

# 8. Save Artifacts

</a>

In [ ]:
# Convert sets to sorted lists for JSON serialization
occ_to_skills_json = {occ: sorted(list(skills)) for occ, skills in occ_to_skills.items()}

In [ ]:
# Output directory for ontology artifacts
OUT_DIR = PROJECT_ROOT / "Data" / "Ontology Artifacts"
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
#save_json(occ_to_skills_json, "occ_to_skills.json", out_dir=OUT_DIR)
#save_json(skill_df, "skill_df.json", out_dir=OUT_DIR)
#save_json(skill_idf, "skill_idf.json", out_dir=OUT_DIR)
#save_json(occ_avg_skill_idf, "occ_avg_skill_idf.json", out_dir=OUT_DIR)
#save_json(skill_to_parents, "skill_to_parents.json", out_dir=OUT_DIR)